In [ ]:
# ==== Edit this first ====
REPO_URL = "https://github.com/anshularavind/cs188-default-project.git"
REPO_DIR = "/content/cs188-default-project"

# ==== Training options ====
USE_15M_DIFFUSION = True
EPOCHS = 40
BATCH_SIZE = 64
MAX_EPISODES = None  # e.g. 300 for a faster run

import os
import subprocess
import yaml

def sh(cmd):
    print("+", cmd)
    subprocess.check_call(cmd, shell=True)

# Clone (or refresh) repo
if not os.path.exists(REPO_DIR):
    sh(f"git clone {REPO_URL} {REPO_DIR}")
else:
    sh(f"git -C {REPO_DIR} pull --ff-only")

os.chdir(REPO_DIR)

# Full Colab bootstrap (equivalent essentials from install.sh)
# Includes: apt deps, robosuite/robocasa install, kitchen assets, macros, dataset download, verify
sh("python cabinet_door_project/99_colab_setup.py --download_assets --setup_macros --download_dataset --verify")

os.chdir(os.path.join(REPO_DIR, "cabinet_door_project"))

# Build augmented parquet files (required by default training config)
sh("python 05b_augment_handle_data.py")

config_path = "configs/diffusion_policy.yaml"
if USE_15M_DIFFUSION:
    cfg = yaml.safe_load(open(config_path, "r"))
    # ~15M-ish U-Net preset
    cfg["base_channels"] = 112
    cfg["channel_mults"] = [1, 2, 4]
    cfg["num_res_blocks"] = 2
    cfg["cond_dim"] = 448
    cfg["num_diffusion_iters"] = 100
    cfg["num_inference_iters"] = 16
    config_path = "configs/diffusion_policy_15m_colab.yaml"
    with open(config_path, "w") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    print("Using 15M diffusion config:", config_path)

train_cmd = f"python 06_train_policy.py --config {config_path} --epochs {EPOCHS} --batch_size {BATCH_SIZE}"
if MAX_EPISODES is not None:
    train_cmd += f" --max_episodes {int(MAX_EPISODES)}"
sh(train_cmd)

ckpt = "/tmp/cabinet_policy_checkpoints/best_policy.pt"
sh(f"python 07_evaluate_policy.py --checkpoint {ckpt} --num_rollouts 20 --split pretrain")
sh(f"python 08_visualize_policy_rollout.py --checkpoint {ckpt} --offscreen --video_path /tmp/policy_rollout.mp4")
print("Done. Rollout video: /tmp/policy_rollout.mp4")
